# Ejercicio 3. Consultas directas sobre archivos Parquet

Viajes de taxis amarillos y verdes de enero a agosto de 2026, leídos desde data/raw sin cargarlos a una tabla.

In [1]:
import os
import time
from pathlib import Path

import duckdb
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir("..")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
con = duckdb.connect()


def consulta(ruta):
    """Imprime un archivo de sql, lo ejecuta y muestra el tiempo"""
    texto = Path("sql", ruta).read_text(encoding="utf-8")
    print(texto)
    inicio = time.perf_counter()
    resultado = con.execute(texto).df()
    print(f"-- {time.perf_counter() - inicio:.2f} s")
    return resultado

## 3.1 Archivos disponibles

Cuántos archivos hay por tipo de taxi y qué meses cubren, según los nombres de los archivos en data/raw.

In [2]:
consulta("ej3/01_archivos.sql")

-- Archivos Parquet por tipo de taxi y rango de meses
SELECT
    regexp_extract(file, '(yellow|green)_tripdata', 1) AS tipo,
    count(*) AS archivos,
    min(regexp_extract(file, '(\d{4}-\d{2})\.parquet', 1)) AS primer_mes,
    max(regexp_extract(file, '(\d{4}-\d{2})\.parquet', 1)) AS ultimo_mes
FROM glob('data/raw/*/*/*.parquet')
GROUP BY tipo
ORDER BY tipo

-- 0.03 s


,tipo,archivos,primer_mes,ultimo_mes
0,green,8,2026-01,2026-08
1,yellow,8,2026-01,2026-08


Hay 8 archivos de cada tipo, de enero a agosto de 2026. Septiembre a diciembre no estaban publicados, así que las comparaciones mensuales de 2026 llegan hasta agosto.

## 3.2 Registros disponibles

El pie de cada Parquet guarda su número de registros, así que el conteo no necesita leer filas.

In [3]:
consulta("ej3/02_registros.sql")

-- Registros por tipo de taxi según el pie de cada Parquet, sin leer las filas
WITH archivos AS (
    SELECT
        regexp_extract(file_name, '(yellow|green)_tripdata', 1) AS tipo,
        num_rows,
        num_row_groups
    FROM parquet_file_metadata('data/raw/*/*/*.parquet')
)
SELECT
    coalesce(tipo, 'total') AS tipo,
    count(*) AS archivos,
    sum(num_rows)::BIGINT AS registros,
    sum(num_row_groups)::BIGINT AS grupos_de_filas
FROM archivos
GROUP BY ROLLUP (tipo)
ORDER BY grouping(tipo), tipo

-- 0.04 s


,tipo,archivos,registros,grupos_de_filas
0,green,8,337114,8
1,yellow,8,29703355,32
2,total,16,30040469,40


Son 30,040,469 registros, 29,703,355 de taxis amarillos y 337,114 de verdes. Como los verdes son el 1.1% del total, las comparaciones entre tipos usan porcentajes, promedios y medianas en lugar de conteos.

## 3.3 Columnas presentes

Columnas y tipos de cada tipo de taxi, leyendo todos sus archivos.

In [4]:
consulta("ej3/03_columnas.sql")

-- Columnas y tipos de datos de taxis amarillos y verdes, en el orden de los archivos
WITH yellow AS (
    SELECT row_number() OVER () AS orden, column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true))
),
green AS (
    SELECT row_number() OVER () AS orden, column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true))
)
SELECT
    coalesce(y.column_name, g.column_name) AS columna,
    y.column_type AS tipo_yellow,
    g.column_type AS tipo_green
FROM yellow AS y
FULL JOIN green AS g ON y.column_name = g.column_name
ORDER BY coalesce(y.orden, g.orden + 0.5)

-- 0.01 s


,columna,tipo_yellow,tipo_green
0,VendorID,INTEGER,INTEGER
1,tpep_pickup_datetime,TIMESTAMP,NaN
2,lpep_pickup_datetime,NaN,TIMESTAMP
3,tpep_dropoff_datetime,TIMESTAMP,NaN
4,lpep_dropoff_datetime,NaN,TIMESTAMP
5,passenger_count,BIGINT,BIGINT
6,trip_distance,DOUBLE,DOUBLE
7,RatecodeID,BIGINT,BIGINT
8,store_and_fwd_flag,VARCHAR,VARCHAR
9,PULocationID,INTEGER,INTEGER


Los amarillos tienen 21 columnas y los verdes 22, con 18 en común. Las fechas cambian de prefijo (tpep en amarillos, lpep en verdes), Airport_fee existe en amarillos y ehail_fee y trip_type en verdes. Por eso la vista viajes de sql/vistas.sql renombra las fechas a pickup_datetime y dropoff_datetime y une los dos tipos por nombre de columna.

Falta revisar si todos los archivos de un mismo tipo traen las mismas columnas.

In [5]:
consulta("ej3/04_esquema_por_archivo.sql")

-- Columnas que no están en todos los archivos de un tipo o que cambian de tipo físico
WITH esquema AS (
    SELECT
        regexp_extract(file_name, '(yellow|green)_tripdata', 1) AS tipo,
        regexp_extract(file_name, '(\d{4}-\d{2})\.parquet', 1) AS mes,
        name AS columna,
        type AS tipo_fisico
    FROM parquet_schema('data/raw/*/*/*.parquet')
    WHERE num_children IS NULL
),
totales AS (
    SELECT tipo, count(DISTINCT mes) AS archivos_tipo
    FROM esquema
    GROUP BY tipo
)
SELECT
    e.tipo,
    e.columna,
    count(*) AS archivos_con_columna,
    t.archivos_tipo,
    min(e.mes) AS desde,
    max(e.mes) AS hasta,
    string_agg(DISTINCT e.tipo_fisico, ', ') AS tipos_fisicos
FROM esquema AS e
JOIN totales AS t USING (tipo)
GROUP BY e.tipo, e.columna, t.archivos_tipo
HAVING count(*) < t.archivos_tipo OR count(DISTINCT e.tipo_fisico) > 1
ORDER BY e.tipo, e.columna

-- 0.01 s


,tipo,columna,archivos_con_columna,archivos_tipo,desde,hasta,tipos_fisicos
0,green,request_source,3,8,2026-06,2026-08,BYTE_ARRAY
1,yellow,request_source,3,8,2026-06,2026-08,BYTE_ARRAY


request_source aparece desde junio de 2026 en los dos tipos y no está en enero a mayo. Tampoco figura en el diccionario de datos de la TLC, cuya versión es del 18 de marzo de 2025. La siguiente consulta muestra cómo lee DuckDB esos archivos con y sin union_by_name.

In [6]:
consulta("ej3/05_union_by_name.sql")

-- Columnas que DuckDB expone al leer todos los archivos amarillos con y sin union_by_name
SELECT
    'sin union_by_name' AS lectura,
    count(*) AS columnas,
    list_contains(list(column_name), 'request_source') AS incluye_request_source
FROM (DESCRIBE SELECT * FROM read_parquet('data/raw/yellow/*/*.parquet'))
UNION ALL
SELECT
    'con union_by_name',
    count(*),
    list_contains(list(column_name), 'request_source')
FROM (DESCRIBE SELECT * FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true))

-- 0.00 s


,lectura,columnas,incluye_request_source
0,sin union_by_name,20,False
1,con union_by_name,21,True


Sin union_by_name, DuckDB toma el esquema del primer archivo de la lista, que es enero, y descarta request_source sin avisar. Con el orden inverso, cuando el primer archivo sí tiene la columna, la lectura falla con un error de esquema. Por eso todas las lecturas del proyecto usan union_by_name, y la columna queda nula en los meses que no la traen.

## 3.4 Tipos de datos

Las fechas son TIMESTAMP sin zona horaria, los identificadores de proveedor y de zona son INTEGER, los códigos de tarifa y de pago son BIGINT, los montos y la distancia son DOUBLE, y store_and_fwd_flag y request_source son texto. Los dos tipos de taxi usan el mismo tipo de dato en las columnas que comparten.

RatecodeID, payment_type, VendorID y trip_type son códigos y no cantidades, así que en el ejercicio 4 se agrupan y se traducen con el diccionario de datos en lugar de promediarse.

## 3.5 Muestra de registros

Muestra aleatoria de cada tipo con semilla fija, para que se repita en cada ejecución.

In [7]:
consulta("ej3/06_muestra_yellow.sql")

-- Muestra aleatoria reproducible de viajes amarillos
SELECT *
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)
USING SAMPLE bernoulli(0.00002%) REPEATABLE (42)



-- 1.82 s


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-01-29 10:04:10,2026-01-29 10:09:30,1,0.95,1,N,107,233,1,7.20,0.0,0.5,2.39,0.0,1.0,14.34,2.5,0.0,0.75,NaN
1,2,2026-03-31 09:59:52,2026-03-31 10:08:15,1,1.14,1,N,141,263,1,9.30,0.0,0.5,2.66,0.0,1.0,15.96,2.5,0.0,0.00,NaN
2,2,2026-04-28 14:48:54,2026-04-28 14:54:22,2,0.35,1,N,236,75,1,6.50,0.0,0.5,2.10,0.0,1.0,12.60,2.5,0.0,0.00,NaN
3,2,2026-05-15 08:39:32,2026-05-15 08:51:18,1,1.75,1,N,170,113,1,12.80,0.0,0.5,3.51,0.0,1.0,21.06,2.5,0.0,0.75,NaN
4,7,2026-05-20 19:27:16,2026-05-20 19:27:16,1,1.10,1,N,163,141,2,11.40,0.0,0.5,0.00,0.0,1.0,18.65,2.5,0.0,0.75,NaN
5,2,2026-06-01 14:43:14,2026-06-01 15:13:12,1,2.52,1,N,162,246,2,25.40,0.0,0.5,0.00,0.0,1.0,30.15,2.5,0.0,0.75,NaN
6,2,2026-06-03 16:15:59,2026-06-03 16:20:48,3,0.88,1,N,238,151,2,7.20,2.5,0.5,0.00,0.0,1.0,11.20,0.0,0.0,0.00,NaN
7,2,2026-07-23 11:32:25,2026-07-23 11:46:53,1,0.87,1,N,161,186,1,12.80,0.0,0.5,3.00,0.0,1.0,20.55,2.5,0.0,0.75,NaN
8,2,2026-07-18 12:22:25,2026-07-18 12:34:57,<NA>,1.93,<NA>,NaN,79,231,0,18.69,0.0,0.5,2.34,0.0,1.0,25.78,NaN,NaN,0.75,A
9,2,2026-07-27 05:57:57,2026-07-27 06:07:27,<NA>,2.43,<NA>,NaN,236,230,0,15.65,0.0,0.5,2.00,0.0,1.0,22.40,NaN,NaN,0.75,A


In [8]:
consulta("ej3/07_muestra_green.sql")

-- Muestra aleatoria reproducible de viajes verdes
SELECT *
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true)
USING SAMPLE bernoulli(0.002%) REPEATABLE (42)

-- 0.03 s


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-12 14:40:56,2026-01-12 14:41:36,N,1,82,82,2,0.07,3.00,0.0,0.5,0.0,0.00,NaN,1.0,4.50,2,1,0.00,0.0,NaN
1,6,2026-01-20 07:38:53,2026-01-20 08:26:40,NaN,<NA>,61,125,<NA>,6.51,3.00,0.0,0.5,0.0,0.00,NaN,0.3,35.08,<NA>,<NA>,NaN,0.0,NaN
2,2,2026-02-15 07:58:38,2026-02-15 08:07:44,N,1,74,166,1,2.10,12.10,0.0,0.5,2.0,0.00,NaN,1.0,15.60,1,1,0.00,0.0,NaN
3,2,2026-04-06 21:02:15,2026-04-06 21:14:09,N,1,166,238,1,1.66,12.80,1.0,0.5,0.0,0.00,NaN,1.0,18.05,2,1,2.75,0.0,NaN
4,2,2026-05-05 09:10:00,2026-05-05 09:16:00,NaN,<NA>,145,193,<NA>,1.42,14.33,0.0,0.5,1.0,0.00,NaN,1.0,16.83,<NA>,<NA>,NaN,0.0,NaN
5,2,2026-06-11 09:12:37,2026-06-11 09:20:04,N,1,134,197,5,1.80,10.00,0.0,0.5,0.0,0.00,NaN,1.0,11.50,2,1,0.00,0.0,NaN
6,6,2026-06-12 17:52:17,2026-06-12 18:45:00,NaN,<NA>,238,51,<NA>,14.25,3.00,0.0,0.5,0.0,7.46,NaN,0.3,42.41,<NA>,<NA>,NaN,0.0,A
7,2,2026-08-04 09:27:25,2026-08-04 09:32:16,N,1,55,55,1,0.82,7.20,0.0,0.5,0.0,0.00,NaN,1.0,8.70,2,1,0.00,0.0,NaN


La muestra deja ver tres problemas. Hay registros sin passenger_count, RatecodeID ni congestion_surcharge cuando payment_type es 0 en amarillos o nulo en verdes, un viaje amarillo que empieza y termina en el mismo segundo y un viaje verde de 0.07 millas.

## 3.6 Calidad de datos

SUMMARIZE resume cada columna con su rango, sus cuartiles y su porcentaje de nulos.

In [9]:
consulta("ej3/08_resumen_yellow.sql")

-- Tipo, rango, cuartiles y porcentaje de nulos de cada columna de los viajes amarillos
SUMMARIZE SELECT * FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)



-- 5.33 s


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.885739809526567,0.7155277514022342,2,2,2,29703355,0.00
1,tpep_pickup_datetime,TIMESTAMP,2001-01-01 09:23:58,2026-08-31 23:59:59,16867928,2026-04-30 15:29:29.519432,NaN,2026-03-04 01:40:13.389102,2026-04-30 07:16:13.842466,2026-06-26 03:16:28.180201,29703355,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2001-01-01 16:09:38,2026-09-01 20:16:00,16587125,2026-04-30 15:47:08.186216,NaN,2026-03-03 15:49:43.15829,2026-04-30 13:01:21.186809,2026-06-26 06:15:23.125273,29703355,0.00
3,passenger_count,BIGINT,0,9,11,1.2494318488564,0.6529195072996766,1,1,1,29703355,25.98
4,trip_distance,DOUBLE,0.0,328522.2,7217,5.552940149689247,550.6497893231099,1.0219501283943055,1.8548962937777382,3.814870618241699,29703355,0.00
5,RatecodeID,BIGINT,1,99,7,4.527471444398553,18.0009265400668,1,1,1,29703355,25.98
6,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,29703355,25.98
7,PULocationID,INTEGER,1,265,290,161.57793013617484,66.74656703251704,117,161,233,29703355,0.00
8,DOLocationID,INTEGER,1,265,298,161.05139342003622,70.72548569211345,109,162,234,29703355,0.00
9,payment_type,BIGINT,0,5,6,0.8621735154160195,0.6463324345961728,0,1,1,29703355,0.00


In [10]:
consulta("ej3/09_resumen_green.sql")

-- Tipo, rango, cuartiles y porcentaje de nulos de cada columna de los viajes verdes
SUMMARIZE SELECT * FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true)

-- 0.14 s


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,2.328351833504393,1.2771882973738604,2,2,2,337114,0.00
1,lpep_pickup_datetime,TIMESTAMP,2008-12-31 17:35:31,2026-08-31 23:58:28,332702,2026-05-02 13:25:51.096427,NaN,2026-03-05 20:37:29.433322,2026-05-05 01:21:12.043946,2026-06-29 01:33:50.46926,337114,0.00
2,lpep_dropoff_datetime,TIMESTAMP,2008-12-31 23:16:26,2026-09-02 09:39:37,396286,2026-05-02 13:46:38.833759,NaN,2026-03-05 23:28:04.039352,2026-05-05 01:50:13.020122,2026-06-29 02:22:02.902327,337114,0.00
3,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,337114,14.47
4,RatecodeID,BIGINT,1,99,7,1.2549672434183374,1.0015327424296085,1,1,1,337114,14.47
5,PULocationID,INTEGER,1,265,263,97.32761024460568,56.57830275002366,74,75,104,337114,0.00
6,DOLocationID,INTEGER,1,265,266,142.8768458147689,77.24476821098119,75,140,229,337114,0.00
7,passenger_count,BIGINT,0,9,11,1.3006461144694266,0.9481007387491747,1,1,1,337114,14.47
8,trip_distance,DOUBLE,0.0,179830.92,2472,13.349507644298379,880.4035093234567,1.252820267098526,2.066189847997517,3.691934907892409,337114,0.00
9,fare_amount,DOUBLE,-500.0,1676.7,4808,17.014101995170655,17.958627635270645,8.602913031993499,13.16944565334494,19.706476155344056,337114,0.00


Dentro de archivos de 2026 hay fechas de inicio desde 2001 en amarillos y desde 2008 en verdes. Todas las columnas de cobro tienen montos negativos, con una tarifa mínima de -2,555.20 dólares, y hay distancias de hasta 328,522 millas. Cinco columnas tienen el mismo porcentaje de nulos, 25.98% en amarillos y 14.47% en verdes, y ehail_fee está vacía en todos los viajes verdes.

Que los porcentajes coincidan sugiere que los nulos caen en los mismos registros. La siguiente consulta los cruza con la forma de pago.

In [11]:
consulta("ej3/10_nulos_por_pago.sql")

-- Relación entre la forma de pago y los nulos en pasajeros, código de tarifa y recargos
WITH viajes AS (
    SELECT 'yellow' AS taxi, payment_type, passenger_count, RatecodeID, congestion_surcharge
    FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)
    UNION ALL
    SELECT 'green', payment_type, passenger_count, RatecodeID, congestion_surcharge
    FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true)
)
SELECT
    taxi,
    payment_type,
    count(*) AS viajes,
    round(100 * count(*) / sum(count(*)) OVER (PARTITION BY taxi), 2) AS pct_viajes,
    count(*) FILTER (WHERE passenger_count IS NULL) AS sin_passenger_count,
    count(*) FILTER (WHERE RatecodeID IS NULL) AS sin_ratecode,
    count(*) FILTER (WHERE congestion_surcharge IS NULL) AS sin_congestion
FROM viajes
GROUP BY taxi, payment_type
ORDER BY taxi, payment_type



-- 0.09 s


,taxi,payment_type,viajes,pct_viajes,sin_passenger_count,sin_ratecode,sin_congestion
0,green,1,219980,65.25,0,0,0
1,green,2,65921,19.55,0,0,0
2,green,3,1688,0.50,0,0,0
3,green,4,750,0.22,0,0,0
4,green,<NA>,48775,14.47,48775,48775,48775
5,yellow,0,7716688,25.98,7716688,7716688,7716688
6,yellow,1,18941008,63.77,0,0,0
7,yellow,2,2708031,9.12,0,0,0
8,yellow,3,98138,0.33,0,0,0
9,yellow,4,239488,0.81,0,0,0


Todos los nulos están en los viajes con payment_type 0 en amarillos y con payment_type nulo en verdes, y el diccionario define el código 0 como Flex Fare. Esos viajes se conservan porque tienen tarifa y total, pero las métricas de pasajeros y recargos del ejercicio 4 no los incluyen.

Para el resto de problemas se cuentan los registros que rompen reglas de fecha, duración, distancia, montos o zona. El mes de cada archivo sale de su nombre.

In [12]:
inconsistencias = consulta("ej3/11_inconsistencias.sql").set_index("taxi").T
porcentaje = (100 * inconsistencias / inconsistencias.loc["viajes"]).round(3)
pd.concat({"registros": inconsistencias, "porcentaje": porcentaje}, axis=1)

-- Registros que rompen reglas de fecha, duración, distancia, montos o zona
WITH viajes AS (
    SELECT
        'yellow' AS taxi,
        regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
        tpep_pickup_datetime AS inicio,
        tpep_dropoff_datetime AS fin,
        trip_distance, fare_amount, total_amount, passenger_count, PULocationID, DOLocationID
    FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true, filename = true)
    UNION ALL
    SELECT
        'green',
        regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1),
        lpep_pickup_datetime,
        lpep_dropoff_datetime,
        trip_distance, fare_amount, total_amount, passenger_count, PULocationID, DOLocationID
    FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true, filename = true)
)
SELECT
    taxi,
    count(*) AS viajes,
    count(*) FILTER (WHERE strftime(inicio, '%Y-%m') <> mes_archivo) AS inicio_fuera_del_mes,
    count(*) FILTER (WHERE fin <= inicio

-- 0.36 s


registros         porcentaje         
taxi                        yellow   green     yellow    green
viajes                    29703355  337114    100.000  100.000
inicio_fuera_del_mes           146      98      0.000    0.029
duracion_cero_o_negativa    371683     234      1.251    0.069
duracion_mayor_3h            10127    1287      0.034    0.382
distancia_cero              952231   12212      3.206    3.623
distancia_mayor_100           1223      72      0.004    0.021
velocidad_mayor_80            7987    1158      0.027    0.344
tarifa_negativa             157364     999      0.530    0.296
total_cero_o_negativo       167093    1566      0.563    0.465
pasajeros_cero               91359    4527      0.308    1.343
zona_desconocida            201486    5886      0.678    1.746

En amarillos, el 3.21% de los viajes tiene distancia cero, el 1.25% termina antes o en el mismo segundo en que empieza y el 0.56% tiene total cero o negativo. Hay 146 viajes amarillos y 98 verdes que empiezan fuera del mes de su archivo, 1,223 amarillos con más de 100 millas y 201,486 con zona desconocida. En verdes la duración cero o negativa es menor (0.07%), pero pesan más los viajes de más de 3 horas (0.38%), las velocidades mayores a 80 millas por hora (0.34%), los pasajeros en cero (1.34%) y la zona desconocida (1.75%).

Por último, las filas repetidas en todas sus columnas.

In [13]:
consulta("ej3/12_duplicados.sql")

-- Filas idénticas en todas las columnas
-- Primero agrupa por el hash de cada fila, que ocupa poca memoria, y después confirma con todas las columnas
WITH yellow AS NOT MATERIALIZED (
    SELECT f.*, hash(f) AS h
    FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true) AS f
),
green AS NOT MATERIALIZED (
    SELECT f.*, hash(f) AS h
    FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true) AS f
),
repetidas_yellow AS (
    SELECT * EXCLUDE (h), count(*) AS copias
    FROM yellow
    WHERE h IN (SELECT h FROM yellow GROUP BY h HAVING count(*) > 1)
    GROUP BY ALL
    HAVING count(*) > 1
),
repetidas_green AS (
    SELECT * EXCLUDE (h), count(*) AS copias
    FROM green
    WHERE h IN (SELECT h FROM green GROUP BY h HAVING count(*) > 1)
    GROUP BY ALL
    HAVING count(*) > 1
)
SELECT 'yellow' AS taxi, count(*) AS grupos_repetidos, coalesce(sum(copias - 1), 0)::BIGINT AS filas_sobrantes
FROM repetidas_yellow
UNION ALL
SELECT 'green', count(*), coalesc

-- 0.89 s


,taxi,grupos_repetidos,filas_sobrantes
0,yellow,7,7
1,green,0,0


Siete filas amarillas aparecen dos veces y no hay repetidas en verdes.

Con estos resultados, sql/vistas.sql define la vista viajes_validos. Conserva los viajes que empiezan en el mes de su archivo, terminan después de empezar, duran hasta 3 horas, recorren más de 0 y hasta 100 millas, no pasan de 80 millas por hora, no tienen tarifa negativa y tienen total mayor que cero. Las siete filas repetidas se dejan porque no cambian ningún resultado, y las zonas 264 y 265 quedan como desconocidas.

In [14]:
vistas = Path("sql/vistas.sql").read_text(encoding="utf-8")
print(vistas)
con.execute(vistas)
consulta("ej3/13_viajes_validos.sql")

-- Vistas sobre los Parquet de data/raw, sin copiar datos

-- Viajes amarillos y verdes con nombres comunes y periodo como el mes del archivo de origen
CREATE OR REPLACE VIEW viajes AS
SELECT
    'yellow' AS taxi,
    strptime(regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1), '%Y-%m')::DATE AS periodo,
    tpep_pickup_datetime AS pickup_datetime,
    tpep_dropoff_datetime AS dropoff_datetime,
    * EXCLUDE (tpep_pickup_datetime, tpep_dropoff_datetime, filename)
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true, filename = true)
UNION ALL BY NAME
SELECT
    'green' AS taxi,
    strptime(regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1), '%Y-%m')::DATE AS periodo,
    lpep_pickup_datetime AS pickup_datetime,
    lpep_dropoff_datetime AS dropoff_datetime,
    * EXCLUDE (lpep_pickup_datetime, lpep_dropoff_datetime, filename)
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true, filename = true);

-- Viajes que cumplen las reglas de calidad defin

-- 0.27 s


,taxi,registros,validos,excluidos,pct_validos
0,yellow,29703355,28223766,1479589,95.02
1,green,337114,321395,15719,95.34


La vista conserva el 95.02% de los registros amarillos y el 95.34% de los verdes.

## 3.7 y 3.8 Consultas sobre los archivos

Las trece consultas están en sql/ej3 y leen los Parquet con glob, parquet_file_metadata, parquet_schema, read_parquet o vistas sobre read_parquet. No se creó ninguna tabla y la conexión de DuckDB vive en memoria.

En esta ejecución los patrones de data/raw cubren los 16 archivos de 2026, 8 amarillos y 8 verdes.

| Consulta | Objetivo | Fuente | Resultado | Decisión |
| --- | --- | --- | --- | --- |
| 01_archivos.sql | Contar archivos y meses por tipo | glob sobre los 16 archivos | 8 archivos por tipo, de enero a agosto de 2026 | Las comparaciones mensuales de 2026 llegan hasta agosto |
| 02_registros.sql | Contar registros sin leer filas | parquet_file_metadata de los 16 archivos | 30,040,469 registros, 98.9% amarillos | Comparar tipos con porcentajes, promedios y medianas |
| 03_columnas.sql | Listar columnas y tipos de datos | read_parquet de los 8 archivos de cada tipo | 21 columnas en amarillos y 22 en verdes, 18 en común | Renombrar las fechas y unir los tipos por nombre de columna en la vista viajes |
| 04_esquema_por_archivo.sql | Encontrar columnas que no están en todos los archivos | parquet_schema de los 16 archivos | request_source aparece desde junio | Revisar la lectura con y sin union_by_name |
| 05_union_by_name.sql | Ver qué columnas expone DuckDB con y sin union_by_name | read_parquet de los 8 archivos amarillos | 20 columnas sin la opción y 21 con ella | Leer siempre con union_by_name |
| 06_muestra_yellow.sql y 07_muestra_green.sql | Ver registros reales | read_parquet de cada tipo con semilla 42 | Nulos en viajes Flex Fare, un viaje de cero segundos y otro de 0.07 millas | Revisar nulos, duración y distancia |
| 08_resumen_yellow.sql y 09_resumen_green.sql | Rango, cuartiles y nulos de cada columna | read_parquet de cada tipo | Fechas desde 2001, montos negativos, distancias de hasta 328,522 millas y el mismo porcentaje de nulos en cinco columnas | Cruzar los nulos con la forma de pago y contar inconsistencias |
| 10_nulos_por_pago.sql | Relacionar los nulos con la forma de pago | read_parquet de los 16 archivos | Todos los nulos están en viajes Flex Fare | Conservar esos viajes y dejarlos fuera de las métricas de pasajeros y recargos |
| 11_inconsistencias.sql | Contar registros que rompen reglas de fecha, duración, distancia, montos o zona | read_parquet de los 16 archivos con el nombre de cada uno | 3.21% de los amarillos con distancia cero y 1.25% con duración cero o negativa, entre otros | Definir las reglas de viajes_validos |
| 12_duplicados.sql | Contar filas repetidas | read_parquet de los 16 archivos | 7 filas amarillas aparecen dos veces y ninguna verde | Conservarlas porque no cambian los resultados |
| 13_viajes_validos.sql | Medir lo que conserva viajes_validos | Vistas viajes y viajes_validos sobre los 16 archivos | 95.02% de los amarillos y 95.34% de los verdes | Usar viajes_validos en el ejercicio 4 |

## 3.9 Consultar un Parquet sin importarlo

DuckDB lee el archivo en el momento de la consulta, sin cargarlo antes a una tabla. Un Parquet guarda los datos por columnas, en grupos de filas, con un pie que tiene el esquema, el número de registros y el mínimo y máximo de cada columna en cada grupo. Con eso DuckDB lee las columnas que usa la consulta y omite las demás, salta los grupos de filas que no cumplen el filtro, responde conteos y esquemas desde el pie, como en 3.2, y reparte el trabajo entre los núcleos del procesador.

Con volúmenes grandes eso evita copiar los datos a otro formato y esperar una carga, no exige que el conjunto quepa en memoria y deja cualquier archivo nuevo de data/raw disponible en la siguiente consulta. El costo de cada consulta depende de las columnas y grupos que lee y no del tamaño total de los archivos. SUMMARIZE recorrió las 21 columnas de 29.7 millones de viajes en segundos.